# Classificação de Imagens com MobileNetV3

Neste Notebook vamos iterar sobre **todas as imagens** na sua pasta de testes e utilizar o modelo super leve e rápido `mobilenetv3_small_100.lamb_in1k` para classificá-las usando apenas a CPU da sua máquina.

In [ ]:
import os
import urllib.request
import torch
from PIL import Image
import timm
from torchvision import transforms
import matplotlib.pyplot as plt

# 1. Definir a pasta das imagens
PASTA_IMAGENS = "C:/Users/luiz/Pictures/testes/"

# 2. Carregar classes da ImageNet
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
classes_file = "imagenet_classes.txt"
if not os.path.exists(classes_file):
    urllib.request.urlretrieve(url, classes_file)

with open(classes_file, "r") as f:
    categories = [s.strip() for s in f.readlines()]

# 3. Carregar modelo e definir transforms
model = timm.create_model('mobilenetv3_small_100.lamb_in1k', pretrained=True)
model.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### Iterando as imagens de `PASTA_IMAGENS` e Classificando...

In [ ]:
print(f"\n🔎 Vasculhando imagens na pasta: {PASTA_IMAGENS}\n...")

if not os.path.exists(PASTA_IMAGENS):
    print("Pasta de imagens não encontrada! Verifique o caminho.")
else:
    for filename in os.listdir(PASTA_IMAGENS):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(PASTA_IMAGENS, filename)
            input_image = Image.open(img_path).convert('RGB')
            input_tensor = transform(input_image)
            input_batch = input_tensor.unsqueeze(0)
            
            with torch.no_grad():
                output = model(input_batch)
            
            probabilities = torch.nn.functional.softmax(output[0], dim=0)
            top5_prob, top5_catid = torch.topk(probabilities, 5)
            
            best_cat = categories[top5_catid[0]]
            best_prob = top5_prob[0].item() * 100
            
            plt.figure(figsize=(4, 4))
            plt.imshow(input_image)
            plt.title(f"{filename}\n{best_cat} ({best_prob:.1f}%)")
            plt.axis('off')
            plt.show()